## Silver — `municipios` (DTB)

**Origem:** `workspace.bronze.dtb` → **Destino:** `workspace.silver.municipios`

- **Grão:** 1 linha por município, deduplicado por `codigo_municipio` (código IBGE de 7 dígitos com DV).
- **Transformações:**
  - Filtra linhas com PK inválida: `codigo_municipio` deve casar com `^[0-9]{7}$`.
  - `codigo_municipio` = trim de `codigo_municipio_completo` (string limpa vinda do `read_ods`, sem artefato `.0`).
  - `codigo_uf` = pad esquerdo (2 dígitos) da coluna bronze `uf` (**código** da UF, não sigla).
  - `sigla_uf` = derivada de `codigo_uf` via mapa oficial IBGE código → sigla (a sigla NÃO existe no arquivo origem).
  - Códigos de região geográfica intermediária/imediata = pad esquerdo (4/6 dígitos) para preservar formato oficial.
  - Colunas de nome (`nome_uf`, `nome_municipio`, nomes de região) = trim + colapso de espaços (casing original do IBGE é preservado — já vem em Title Case).
  - Validação auxiliar: campo bronze `municipio` deve ser prefixo (6 primeiros dígitos) de `codigo_municipio`.
  - `dropDuplicates(["codigo_municipio"])` garante unicidade da PK.
- **Linhagem:** ODS (header linha 7) → `bronze.dtb` → limpeza/padronização → `silver.municipios`.

In [0]:
%run ./_setup_dtb

In [0]:
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from metadata.metadata import SILVER_MUNICIPIOS_COMMENTS

In [0]:
SOURCE_TABLE = "workspace.bronze.dtb"
TARGET_TABLE = "workspace.silver.municipios"

# Mapa oficial IBGE: código UF (2 díg) -> (sigla, nome, grande região)
UF_MAP = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF",
}

# Contrato de saída silver.municipios
COLUNAS_FINAIS = [
    "codigo_municipio",
    "codigo_uf",
    "sigla_uf",
    "nome_uf",
    "codigo_regiao_geografica_intermediaria",
    "nome_regiao_geografica_intermediaria",
    "codigo_regiao_geografica_imediata",
    "nome_regiao_geografica_imediata",
    "nome_municipio",
]

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
print(f"Bronze: {df_bronze.count():,} linhas | colunas: {df_bronze.columns}")
display(df_bronze.limit(5))

In [0]:
def limpar_texto(col):
    """Trim + colapso de espaços; casing original do IBGE é mantido."""
    return F.trim(F.regexp_replace(F.trim(col.cast("string")), r"\s+", " "))


def pad_codigo(col, width):
    """String numérica limpa + zero-pad à esquerda (formato oficial dos códigos)."""
    return F.lpad(F.trim(col.cast("string")), width, "0")


df = df_bronze

# 1) PK do município: código completo de 7 dígitos; descarta linhas inválidas
df = df.withColumn(
    "codigo_municipio",
    F.trim(F.col("codigo_municipio_completo").cast("string"))
)
invalidas = df.filter(~F.col("codigo_municipio").rlike("^[0-9]{7}$")).count()
if invalidas:
    print(f"Aviso: {invalidas} linhas com codigo_municipio_completo inválido serão descartadas")
df = df.filter(F.col("codigo_municipio").rlike("^[0-9]{7}$"))

# 2) Código e nome da UF
df = df.withColumn("codigo_uf", pad_codigo(F.col("uf"), 2))
df = df.withColumn("nome_uf", limpar_texto(F.col("nome_uf")))

# 3) Sigla da UF via mapa código -> sigla (sigla não existe no arquivo origem)
sigla_expr = None
for codigo, sigla in UF_MAP.items():
    cond = F.col("codigo_uf") == codigo
    sigla_expr = F.when(cond, F.lit(sigla)) if sigla_expr is None else sigla_expr.when(cond, F.lit(sigla))
df = df.withColumn("sigla_uf", sigla_expr)

# 4) Regiões geográficas: códigos pad (4/6 díg) + nomes limpos
df = df.withColumn("codigo_regiao_geografica_intermediaria", pad_codigo(F.col("regiao_geografica_intermediaria"), 4))
df = df.withColumn("nome_regiao_geografica_intermediaria", limpar_texto(F.col("nome_regiao_geografica_intermediaria")))
df = df.withColumn("codigo_regiao_geografica_imediata", pad_codigo(F.col("regiao_geografica_imediata"), 6))
df = df.withColumn("nome_regiao_geografica_imediata", limpar_texto(F.col("nome_regiao_geografica_imediata")))

# 5) Nome do município
df = df.withColumn("nome_municipio", limpar_texto(F.col("nome_municipio")))

# 6) Conciliação: 'municipio' (6 díg, sem DV) deve ser prefixo da PK completa
divergentes = (
    df.filter(
        F.substring(F.col("codigo_municipio"), 1, 6)
        != F.lpad(F.col("municipio"), 6, "0")
    )
    .count()
)
print(f"Conciliação municipio vs codigo_municipio: {'OK' if divergentes == 0 else f'{divergentes} divergências'}")

# 7) Seleciona contrato final e deduplica pela PK
df = df.select(*COLUNAS_FINAIS).dropDuplicates(["codigo_municipio"])
print(f"Silver: {df.count():,} municípios após deduplicação")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    SILVER_MUNICIPIOS_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos = spark.table(TARGET_TABLE).select("codigo_municipio").distinct().count()
sem_uf = spark.table(TARGET_TABLE).filter(F.col("codigo_uf").isNull() | F.col("sigla_uf").isNull()).count()
sem_regiao = spark.table(TARGET_TABLE).filter(
    F.col("codigo_regiao_geografica_intermediaria").isNull()
    | F.col("codigo_regiao_geografica_imediata").isNull()
).count()
print(f"Total: {total:,} | PK distintos: {distintos:,} | Duplicatas PK: {total - distintos:,}")
print(f"Linhas sem UF/sigla: {sem_uf:,} | Linhas sem região: {sem_regiao:,}")
display(spark.sql(f"SELECT sigla_uf, nome_uf, count(*) AS qtd_municipios FROM {TARGET_TABLE} GROUP BY sigla_uf, nome_uf ORDER BY sigla_uf"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))